# SMS Spam Classifier — Complete Project Notebook

This notebook walks through the complete project: dataset inspection, preprocessing, training-only cross-validation, validation threshold selection, one final held-out test evaluation, metrics, plots, local predictions, and artifact integrity checks.

**Run from the project root** after installing `requirements.txt`. The training cell takes a short time and regenerates the model, metrics, plots, and project reports. It uses the checked-in UCI SMS Spam Collection; it does not download data or add synthetic training examples.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "spam_classifier").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "spam_classifier").is_dir():
    raise RuntimeError("Open this notebook from the project root or its notebooks/ folder.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

print(f"Project root: {PROJECT_ROOT}")

## 1. Inspect the dataset and split sizes

The project uses the official UCI SMS Spam Collection. Exact duplicates and identical message text with conflicting labels are handled before splitting. The split is stratified 70/15/15 with seed 42. Here we show row counts only; model fitting and held-out evaluation happen in the training workflow below.

In [ ]:
from spam_classifier.config import DATA_PATH
from spam_classifier.data import load_dataset, stratified_split

frame, dataset_info = load_dataset(DATA_PATH)
train_frame, validation_frame, test_frame = stratified_split(frame)

dataset_summary = pd.DataFrame([
    {"item": "Raw messages", "count": dataset_info["raw_rows"]},
    {"item": "Exact duplicates removed", "count": dataset_info["exact_duplicate_rows_removed"]},
    {"item": "Conflicting texts removed", "count": dataset_info["conflicting_message_texts_removed"]},
    {"item": "Rows after cleaning", "count": len(frame)},
    {"item": "Training rows (70%)", "count": len(train_frame)},
    {"item": "Validation rows (15%)", "count": len(validation_frame)},
    {"item": "Held-out test rows (15%)", "count": len(test_frame)},
])
display(dataset_summary)
print("Dataset SHA-256:", dataset_info["sha256"])

## 2. See the offline preprocessing

Preprocessing uses Unicode normalization, contact and number placeholders, a regex tokenizer, and scikit-learn's built-in English stop-word set. It has no runtime downloads. The log redactor is shown separately because it sanitizes text before persistence.

In [ ]:
from spam_classifier.preprocess import normalize_text, tokenize_text, redact_for_log

sample = "Email test@example.org or visit example.com/claim?token=demo123. Call +1 (415) 555-0199."
print("Normalized for the model:")
print(normalize_text(sample))
print("\nTokens:")
print(tokenize_text(normalize_text(sample)))
print("\nRedacted for default logs:")
print(redact_for_log(sample))

## 3. Train and select the model

The trainer compares Count and TF-IDF features, unigram and bigram ranges, stop-word choices, Multinomial Naive Bayes, and Logistic Regression variants. Five-fold cross-validation runs on the training partition only. It then tunes the spam threshold on validation data to maximize recall while requiring at least 0.98 spam precision. The held-out test partition is evaluated once, at the end.

In [ ]:
from spam_classifier.train import run_training

metrics = run_training()
print("Selected model:", metrics["selected_model"]["name"])
print("Selected threshold:", metrics["threshold"]["value"])
print("Validation precision:", metrics["threshold"]["validation_precision"])
print("Validation recall:", metrics["threshold"]["validation_recall"])
print("Validation precision constraint met:", metrics["threshold"]["constraint_met"])

## 4. Compare the experiments and review held-out metrics

Cross-validation results below come from the training split. The classification report, confusion matrix, PR-AUC, and ROC-AUC come from the held-out test split after model and threshold selection.

In [ ]:
comparison = pd.DataFrame(metrics["model_comparison"])
comparison_columns = [
    "name",
    "precision_spam_mean",
    "precision_spam_std",
    "recall_spam_mean",
    "f1_spam_mean",
]
display(comparison[comparison_columns].head(10).round(4))

test_classes = pd.DataFrame.from_dict(
    {label: metrics["test"][label] for label in ("ham", "spam")},
    orient="index",
)
display(test_classes[["precision", "recall", "f1-score", "support"]].round(4))
print(f"Test accuracy: {metrics['test']['accuracy']:.4f}")
print(f"Test PR-AUC: {metrics['test']['pr_auc']:.4f}")
print(f"Test ROC-AUC: {metrics['test']['roc_auc']:.4f}")
print("Confusion matrix (rows=actual ham, spam; columns=predicted ham, spam):")
print(metrics["test"]["confusion_matrix"])

## 5. View the evaluation plots

The trainer saves these plots under `artifacts/` so they can also be opened outside the notebook.

In [ ]:
display(Image(filename=str(PROJECT_ROOT / "artifacts" / "confusion_matrix.png")))
display(Image(filename=str(PROJECT_ROOT / "artifacts" / "precision_recall_curve.png")))

## 6. Try local predictions

These examples demonstrate inference only; they are not training data or evaluation labels. The final loan-offer example may be classified as ham because the training corpus is older and English-only. Treat outputs as model decisions, not verified ground truth.

In [ ]:
from spam_classifier.predict import predict_message

examples = [
    "Congratulations! You won a free prize, click now",
    "Meeting moved to 3 pm. See you at the office.",
    "Get an instant loan without any documents. Apply now!",
]
predictions = []
for message in examples:
    result = predict_message(message)
    predictions.append({
        "message": message,
        "label": result["label"],
        "spam_probability": result["spam_probability"],
        "threshold": result["threshold"],
    })
display(pd.DataFrame(predictions).round({"spam_probability": 4, "threshold": 4}))

## 7. Verify the local artifact bundle

Prediction checks the fixed artifact bundle against its SHA-256 manifest before loading the model. This cell verifies and reports the bundle members without printing message data.

In [ ]:
from spam_classifier.security import verified_artifact_bytes

verified = verified_artifact_bytes()
display(pd.DataFrame([
    {"artifact": name, "verified_bytes": len(content)}
    for name, content in verified.items()
]))

## Notes and limitations

- Re-run the training cell to regenerate model artifacts and reports from the local dataset.
- The dataset is old and English-only; modern regional-language, OTP, KYC, UPI, and loan scams may be missed.
- Do not load untrusted joblib files. The SHA-256 manifest is not a digital signature.
- Default prediction logs redact contact details and URLs; raw message logging requires the explicit `--log-raw` option.

For setup, command-line usage, full metrics, and security results, see the project `README.md` and `docs/`.